In [1]:
import wandb
api = wandb.Api()
import numpy as np

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
import json

In [4]:
runs = api.runs("ramithx1/Batch-Run-Maximum-Parsimony-A100-META-optimized")

In [5]:
def calc_avg_seq_similarity(leaves_data):
    length = leaves_data.shape[1]
    sim_percentages = []
    
    for i in range(0, leaves_data.shape[0]):
        for j in range(i+1, leaves_data.shape[0]):
            sim = (leaves_data[i] == leaves_data[j]).sum()/length
            
            sim_percentages.append(sim)
    
    sim_percentages = np.array(sim_percentages)
    
    return sim_percentages.mean()

In [18]:
def retrieve_exp(special_word, num_mutations):
    main_dict = {}
    errors = {}

    simulated = {}
    sankoffs = {}
    elves = {}

    for l in [4,8,16,32,64,128]:
        errors[str(l)] = []
        sankoffs[str(l)] = []
        elves[str(l)] = []
        simulated[str(l)] = []
        main_dict[str(l)] = []

        for run in runs:
            n_leaves = run.config['n_leaves']
            n_letters = run.config['n_letters']
            seq_length = run.config['seq_length']

            if(n_leaves == l and run.config['tLs']==[0, 0.005, 10, 50]):

                if("custom" in run.name or 'best cost' not in run.summaryMetrics.keys()):
                    continue

                if(special_word not in run.name or run.config['n_mutations'] != num_mutations):
                    continue

                assert n_letters == 20

                # print(run.config['epochs'])
                temp = run
                cost = run.summaryMetrics['best cost'] #run.summaryMetrics['traversal cost']
                gt   = run.summaryMetrics['sankoff_cost']
                sim = run.summaryMetrics['Groundtruth Total Loss']

                plotly_plot = run.summaryMetrics['Groundtruth Seq']
                
                
                if not os.path.exists(plotly_plot['path']):
                    run.file(plotly_plot['path']).download(replace=True) ## do just once

                with open(plotly_plot['path'], 'r') as file:
                    seq_data = np.array(json.load(file)['data'][0]['z'])

                    assert 2*n_leaves - 1 == seq_data.shape[0]
                    assert seq_length     == seq_data.shape[1]

                    leaves_data = seq_data[:n_leaves,:]

                val = calc_avg_seq_similarity(leaves_data)
                # print(f"{n_leaves} --> {val}")

                main_dict[str(l)].append([val, (cost - gt)/gt*100])

                errors[str(l)].append(cost - gt)
                sankoffs[str(l)].append(gt)
                elves[str(l)].append(cost)
                simulated[str(l)].append(sim)


                # print(run.summaryMetrics.keys())
                # print(run.summaryMetrics['sankoff_cost'])
                # print(run.summaryMetrics['traversal cost'])
                # print(run.history)
    
    for key in main_dict.keys():
        print(f"{key} --> {len(main_dict[key])}")
    
    return main_dict


### Diverse 50 Mutation Experiments

In [23]:
main_dict = retrieve_exp("Diverse!", num_mutations = 50)

4 --> 10
8 --> 10
16 --> 10
32 --> 10
64 --> 10
128 --> 0


In [24]:
for key in [4,8,16,32,64]:
    
    sim    = np.array([x[0] for x in main_dict[str(key)]])
    errors = np.array([x[1] for x in main_dict[str(key)]])
    

    print(f"Leaves \t {key} \t Avg Similarity = \t {round(sim.mean(),3)}±{round(sim.std(),2)} \t Avg Error = \t {round(errors.mean(),3)}±{round(errors.std(),2)}")     

Leaves 	 4 	 Avg Similarity = 	 0.518±0.01 	 Avg Error = 	 0.0±0.0
Leaves 	 8 	 Avg Similarity = 	 0.394±0.01 	 Avg Error = 	 1.313±0.86
Leaves 	 16 	 Avg Similarity = 	 0.283±0.01 	 Avg Error = 	 5.036±1.15
Leaves 	 32 	 Avg Similarity = 	 0.195±0.01 	 Avg Error = 	 8.586±1.37
Leaves 	 64 	 Avg Similarity = 	 0.138±0.01 	 Avg Error = 	 11.488±0.98


### Diverse 10 Mutation Experiments

In [25]:
main_dict = retrieve_exp("Diverse!M10", num_mutations = 10)

for key in [4,8,16,32,64]:
    
    sim    = np.array([x[0] for x in main_dict[str(key)]])
    errors = np.array([x[1] for x in main_dict[str(key)]])
    

    print(f"Leaves \t {key} \t Avg Similarity = \t {round(sim.mean(),3)}±{round(sim.std(),2)} \t Avg Error = \t {round(errors.mean(),3)}±{round(errors.std(),2)}")     

4 --> 10
8 --> 10
16 --> 10
32 --> 10
64 --> 10
128 --> 0
Leaves 	 4 	 Avg Similarity = 	 0.88±0.0 	 Avg Error = 	 0.0±0.0
Leaves 	 8 	 Avg Similarity = 	 0.829±0.0 	 Avg Error = 	 0.5±1.5
Leaves 	 16 	 Avg Similarity = 	 0.762±0.01 	 Avg Error = 	 7.188±2.73
Leaves 	 32 	 Avg Similarity = 	 0.676±0.02 	 Avg Error = 	 14.806±3.49
Leaves 	 64 	 Avg Similarity = 	 0.594±0.03 	 Avg Error = 	 26.864±2.49


In [26]:
main_dict

{'4': [[0.8802083333333334, 0.0],
  [0.8756510416666666, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0],
  [0.8802083333333334, 0.0]],
 '8': [[0.8310546875, 0.0],
  [0.8226841517857143, 5.0],
  [0.8304966517857143, 0.0],
  [0.8304966517857143, 0.0],
  [0.8304966517857143, 0.0],
  [0.8304966517857143, 0.0],
  [0.8304966517857143, 0.0],
  [0.8351004464285714, 0.0],
  [0.8321707589285714, 0.0],
  [0.8184988839285714, 0.0]],
 '16': [[0.7513020833333334, 11.37123745819398],
  [0.762890625, 7.744107744107744],
  [0.7654296875, 5.704697986577181],
  [0.76455078125, 4.682274247491638],
  [0.75927734375, 11.333333333333332],
  [0.755859375, 9.666666666666666],
  [0.7720377604166667, 7.333333333333333],
  [0.7626627604166667, 2.6755852842809364],
  [0.7671549479166667, 5.68561872909699],
  [0.7619791666666667, 5.68561872909699]],
 '32': 